# Lesson 1 · No black boxes — why Aurora exists, where it's headed, and a health check for your setup

**Start here (read this first)**: you're not going to write Whisper in one sitting. First, get three things on solid ground — **why we build from scratch**, **where the next 6 months will take you**, and **whether your environment actually runs**. The sine wave is just the opening "first sound": a quick first feel for what sound looks like once it's inside a computer.

**Learning goals**
1. Understand why Aurora implements things from scratch instead of calling ready-made APIs
2. See the full 11-module roadmap and the milestone that marks each month
3. Internalize what each module is for, using the pattern "I'm learning X so I can do Y"
4. Implement `check_imports` and `environment_report` — debug your dev environment proactively
5. Write down 3 personal goals you can come back and check in 6 months

### ⏱ The 30-minute "minimum path" for this lesson (take this route when you're short on time)

The sections below pack in a lot — think of them as a big fold-out map. If you just want to find your footing before the course really starts, follow this order and leave the rest folded up in the map; you can come back to it any time:

1. Read **§1 Why write it from scratch?** + **the three keys**, and get the sine-wave cell running (build the picture)
2. Skim **§2 the principle**, **§3 the four systems**, **§4 the roadmap** (know where you're going; no need to memorize)
3. Do the entries in **§6 what each module means** that you can already write + **§11 three personal goals**
4. Implement **`check_imports`** and **`environment_report`**, and get the smoke test passing

§5 the milestone table, §7 the interview follow-up table, §8 the shortcuts: treat them as "trailers / tool cards" for now. Nobody expects you to understand all of it on day one.


← **Starting point** · no previous lesson

> This lesson sets up Aurora's motivation, its roadmap, and a set of tools for checking your environment.
> The next lesson moves on to **sampling and your first audible sine wave** (L02) — turning the sample rate you "glimpsed" here into four small functions you write yourself.


## 1. Why write it from scratch?

Two lines of code are all it takes to call Whisper as a black box:

```python
import openai
openai.audio.transcriptions.create(model="whisper-1", file=audio_file)
```

But that doesn't mean you really understand it. When someone presses you on it — an interviewer, a colleague, or your future self — you need to have answers:

| The follow-up question | Can you answer it? |
|---|---|
| How many triangular filters does Whisper's Mel filterbank use? Why 80? | ❌ Not if you only used the API |
| Why is the FFT's time complexity O(N log N)? What is a butterfly network? | ❌ |
| How does CTC loss handle an unknown alignment? What is the forward-backward algorithm? | ❌ |

**One more word on where the line sits**: even if you swap the OpenAI call above for a local `pipeline` or a local inference script, as long as the core algorithm is a black box someone else wrote, you'll still get stuck when an interviewer asks "why?" or "how is that computed?". What makes something a black box isn't the brand name; it's whether you could derive it starting from a blank file.

Aurora's approach: **write every layer yourself**, then check your answers against a reference implementation.
Six months from now, you'll be able to derive the answers to all three of those questions on a whiteboard.


### Before your first piece of code: three keys (it's completely normal not to follow the formulas yet)

The 5 lines of code below are the only "real code" this lesson asks you to run right away. **For now, your only job is to run it and get a feel for the plot** —
L02 takes sampling apart line by line, and sin and 2π are the stars of L04. Here you just get three keys to start with.
You don't need to work everything out; being able to read "what each line is doing" is enough.

**Key 1: `sr = 16000` (sample rate) — 16,000 "snapshots" per second**

Sound changes continuously, but a computer can only store a sequence of separate numbers, so it takes a snapshot every short interval,
like shooting a dance in burst mode. The number of snapshots per second is called the **sample rate**.
`sr = 16000` means 16,000 numbers per second. (Why 16000, and how fast is fast enough? L02 covers that in detail.)

**Key 2: `t = np.arange(round(duration * sr)) / sr` — number the points first, then convert to seconds**

Read this line in two steps:
1. `np.arange(round(0.02 * 16000))` → produces the integer indices `0, 1, 2, ..., 319`.
   0.02 s × 16,000 points per second = 320 points in total, so the indices run up to 319.
2. `/ sr` → converts "which point" into "which moment, in seconds": point n happens at n/16000 seconds.
   So `t` is an evenly spaced sequence of moments: 0 s, 1/16000 s, 2/16000 s, …

**Key 3: `np.sin(2 * np.pi * freq * t)` — 2π means "one full turn"**

In high school you mostly meet angles in **degrees** first (e.g. $\sin 30° = 0.5$); you learned radians too, but plenty of people rarely use them.
`np.sin`, though, takes a different unit of angle — the **radian**: one full turn = $360°$ = $2\pi$ radians (about 6.28).

Picture a point going around a circle at a steady speed; the sin value is its height above the ground. In one full turn, the height rises and falls exactly once —
that's one complete wave. Now read the formula:
- `freq * t` = "how many turns it has made by time t" (it makes freq = 440 turns per second);
- multiplying by `2 * np.pi` = converting "number of turns" into the "number of radians" that np.sin understands.

So $\sin(2\pi \cdot 440 \cdot t)$ reads as: **how the height of a point turning 440 times per second changes over time** — and that's a 440 Hz sound.

By the way: **A4** is the standard A (la) in the middle of the piano — the note orchestras tune to, at 440 Hz.

> Keep these three keys in your pocket; L02 and L04 will take them apart one at a time and explain them again. Now run the code below and look at the plot.


In [1]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
# You can build the sine wave at the very bottom of Whisper's stack in 5 lines of NumPy
import numpy as np
import matplotlib.pyplot as plt

sr, duration, freq = 16000, 0.02, 440  # sample rate, duration (s), the A4 note
t = np.arange(round(duration * sr)) / sr
x = np.sin(2 * np.pi * freq * t)  # this one line is the heart of aurora.audio.sine

plt.figure(figsize=(8, 2))
plt.plot(t * 1000, x)
plt.xlabel('Time (ms)'); plt.ylabel('Amplitude')
plt.title(f'440 Hz sine wave  ({len(t)} samples)')
plt.tight_layout(); plt.show()
print(f'shape={x.shape}  dtype={x.dtype}  range=[{x.min():.2f}, {x.max():.2f}]')

In [ ]:
# Checking the three keys live — print it instead of "just taking it on trust"
# (reuses the t and x already computed in the previous cell)

# Checking key 2: what does t actually look like?
print('First 5 elements of t:', t[:5])
print(f'Gap between neighboring moments: {t[1] - t[0]:.10f} s  (exactly 1/sr = 1/{sr})')
print(f'Total number of points: {len(t)}  (= duration × sample rate = {duration} × {sr})')

# Checking key 3: how many complete waves should fit into 0.02 s?
print(f'\nTheoretical number of complete waves: freq × duration = {freq} × {duration} = {freq * duration:.1f} waves')
peaks = np.sum((x[1:-1] > x[:-2]) & (x[1:-1] > x[2:]))
print(f'Peaks actually counted in the array: {peaks}  — consistent with the theoretical ≈8.8 (take another look at the plot above)')

## 2. Aurora's core principle

> **When you implement a core capability, build the algorithm itself — don't import a black box that already does it.**

Where exactly the line is drawn:

| Module | Allowed | Not allowed |
|---|---|---|
| audio_core | NumPy (array container + elementwise arithmetic) | librosa, scipy.signal |
| deep_learning | PyTorch automatic differentiation (autograd) | `torch.nn` as an algorithmic shortcut |
| ASR / Music | HuggingFace (inference framework) | the Whisper API as proof that "I understand ASR" |
| LLM | HuggingFace transformers | the ChatGPT API as proof that "I implemented an LLM" |

This principle isn't a ritual — it's a line you can explain clearly to anyone (yourself included).

## 3. What you'll build

Common **fake goals** (KPIs that are easy to mistake for progress): 10,000 commits, 20 blog posts, knowing 10 frameworks —
the numbers look good, but they often can't answer "which layer do you actually understand?"
The real goal: **a small number of correct, deep pieces of work you can explain to anyone** — what you come away with is work and thinking you genuinely understand, not a big portfolio count.

Over 6 months, 4 **end-state systems** you can show off (what the whole machine looks like):

```
① Audio Analysis Engine
   FFT + STFT + Mel filterbank + MFCC, all hand-written, 39 tests all green

② Keyword Spotting (KWS) Demo
   CNN classifier, from waveform to yes/no, trained in PyTorch

③ ASR Transcript Engine
   hand-written CTC loss, understanding the Whisper architecture, WER evaluation

④ Podcast Intelligence (RAG + QA)
   TF-IDF retrieval, RAG pipeline, tool-calling agent
```

Each system comes with: a working demo, one technical blog post, and core algorithms you can derive live, any time.

> **How does this line up with the "monthly milestones" in Section 5?**
> ①–④ here are the **finished pieces of work**; Month 1–6 in §5 are **milestones broken down month by month**.
> For example, Month 4's deliverable says "LM Core," meaning: first lay a solid foundation in language models (Transformer / fine-tuning);
> Month 5 then assembles retrieval and question answering into ④ Podcast Intelligence.
> So: **it's not that "the fourth system got renamed to LM"** — it's "before building ④, you lay the Month 4 brick."


## 4. Course roadmap (11 modules)

> **Map-reading tip**: on a first pass, a rough sense of what each term means is enough;
> this lesson only runs into a few abbreviations like FFT and Whisper — every other term gets properly introduced at the start of its own module.

```
L01–L05  ① Opening            motivation, representing sound, spectrogram intuition, trig, complex numbers
              ↓
L06–L08  ② Fourier intuition  Euler's formula → Fourier intuition → visualization
              ↓
L09–L21  ③ Linear algebra     vectors, matrices, eigenvalues, SVD → the DFT seen as a matrix
              ↓
L22–L26  ④ Calculus           derivatives, gradients, the chain rule, gradient descent
              ↓
L27–L31  ⑤ Probability        probability distributions, cross entropy (CE), Softmax
              ↓
L32–L53  ⑥ Audio DSP          FFT ← the single most important stage: hand-write the FFT in L37-L41
              ↓
L54–L65  ⑦ Deep learning (DL)  Autograd → MLP → PyTorch → KWS training
              ↓
L66–L75  ⑧ Automatic speech recognition (ASR)  CTC → Whisper → WER evaluation
              ↓
L76–L82  ⑨ Music              Music embedding → the MusicGen / Suno path
              ↓
L83–L91  ⑩ Language models (LLM)  Transformer → LoRA → RAG → Agent
              ↓
L92–L99  ⑪ Integration        asynchronous pipeline → MLOps → capstone demo
```

**Each layer of understanding is built directly on the one before it; you can't skip ahead.**
The most critical stage: L37-L41 (the hand-written FFT) — the heart of the whole audio_core.

### Matching the two maps: logical modules (①–⑪) vs. folders on disk

①–⑪ above are **logical segments of knowledge**; the folders in `notebooks/` are **where things physically live**,
and the two don't match one-to-one — just as "New England" is a logical region while the states are the actual folders.
When you run the folder check at the end of Section 5 you'll see that `0_foundation` only holds L01–L03 — you didn't misread.
Here's how they line up:

| Folder (the next exercise uses these same names) | Logical modules covered | Lessons |
|---|---|---|
| `0_foundation` | ① Opening (first half: motivation, sound, spectrograms) | L01–L03 |
| `1_complex_trig` | ① Opening (second half: trig, complex numbers) + ② Fourier intuition | L04–L08 |
| `2_linear_algebra` | ③ Linear algebra | L09–L21 |
| `3_calculus` | ④ Calculus | L22–L26 |
| `4_probability` | ⑤ Probability | L27–L31 |
| `5_audio_dsp` | ⑥ Audio DSP | L32–L53 |
| `6_deep_learning` | ⑦ Deep learning | L54–L65 |
| `7_asr` | ⑧ Speech recognition | L66–L75 |
| `8_music` | ⑨ Music | L76–L82 |
| `9_llm` | ⑩ Language models | L83–L91 |
| `10_integration` | ⑪ Integration | L92–L99 |

**One-line takeaway**: there are **11 folders** on disk and, logically, **11 segments of knowledge**. They aren't locked together one-to-one, **but only the very start is a little offset** — the first half of ① lives in `0_foundation`, while the second half of ① shares `1_complex_trig` with ②; ③–⑪ match up essentially one-to-one.
(So yes, you'll see "① Opening" on the roadmap spanning two folders. That's by design, not a sign of missing lessons.)


## 5. Monthly milestones

The roadmap tells you where you're going; the milestones tell you whether you actually got there.

| Month | Milestone | Deliverable |
|---|---|---|
| Month 1 | Can rewrite FFT / STFT / Mel / MFCC from a blank file | Audio Analysis Engine |
| Month 2 | Train a KWS model with test accuracy > 90% | KWS Demo + W&B training curves |
| Month 3 | Can explain CTC forward-backward; hand-write the WER calculation | ASR Transcript Engine |
| Month 4 | Implement Transformer attention from scratch; can fine-tune | LM Core (the language-model foundation for ④) |
| Month 5 | RAG pipeline running end to end; can explain TF-IDF vs. embeddings | Podcast Intelligence (④ fully assembled) |
| Month 6 | Capstone demo running; can derive every core question on a whiteboard | Complete Aurora v1 |

> To pin it down once more: ①–④ in §3 are the four finished pieces; this table breaks the journey into monthly milestones.
> Month 4's "LM Core" ≠ a fifth system started from scratch; it's the brick you must lay before ④ Podcast Intelligence.

At the end of each month, use this table to ask yourself: did I really get there, or did I just click through the code?

> **Panicking at a pile of unfamiliar abbreviations? Stop — you don't need to understand a single term in the table above (or the one in Section 7) yet.
> They aren't "things you should already know"; they're exactly what this course is about to teach,
> and each one is explained from scratch in its own lesson.** The right way to read these two tables is like a movie trailer:
> remember "in 6 months I'll be able to explain all of this," then turn the page. If you'd like an early introduction anyway:
>
> - **W&B** (Weights & Biases): a tool that records the training process — like charting a "temperature curve" for your model's training (used from L57 on)
> - **WER** (Word Error Rate): the score for "how many words did speech recognition get wrong" (L74)
> - **CTC / forward-backward**: the alignment algorithm that lets a model learn even when "it doesn't know which stretch of audio each word belongs to" (L68–L69)
> - **KWS** (Keyword Spotting): wake-word detection — e.g. a light turns on when it hears "yes/no" (the Month 2 project)


In [ ]:
# Check that your course folder structure is complete
from pathlib import Path

# Locate the notebooks root automatically (works wherever the course is installed)
_here = Path.cwd()
nb_root = _here if _here.name != '0_foundation' else _here.parent
modules = sorted([d for d in nb_root.iterdir() if d.is_dir()])

total = 0
for mod in modules:
    nbs = sorted(p for p in mod.glob('*.ipynb') if not p.stem.endswith('.en'))  # skip English copies
    total += len(nbs)
    if nbs:
        nums = [nb.stem.split('_')[0] for nb in nbs]
        print(f'  {mod.name:25s}  {nums[0]}–{nums[-1]}  ({len(nbs)} lessons)')

print(f'\nTotal: {total} notebooks')

## 6. ✏️ Exercise: write down what each module means, in your own words

The roadmap tells you "where you're going"; this exercise tells you "why you're going there."

**Line of reasoning**: for each module, first think about what its input and output are,
then think about what concrete contribution it makes to your Aurora system,
and write it in the pattern `"I'm learning X so that in Aurora I can do Y"`.

Why do this exercise? Because there's a gap between "I understood the roadmap" and "I can explain every step in my own words"
— and interviews test the second one.

### A handhold before you start writing: the 10 keys + a "plain-English line" for each module

The keys of the dictionary below are **folder names** (10 keys — `0_foundation` is the 3 lessons you're reading right now,
so you don't need to write a purpose for it and it isn't in the dictionary; the second half of ① and ② are merged into `1_complex_trig`,
see the table at the end of Section 4). Each key comes with one line of "roughly what it does" as a handhold —
**your task is to connect that line to Aurora and write your own Y, not to copy the roadmap**:

| Dict key | Roadmap module | Lessons | In plain English: roughly what it does |
|---|---|---|---|
| `1_complex_trig` | ① second half + ② | L04–L08 | The math of going around in circles: sin/cos and "numbers with two coordinates" — the language you need to read FFT output |
| `2_linear_algebra` | ③ | L09–L21 | Arranging a pile of numbers into a table and transforming it; the FFT is, at heart, one special kind of "table multiplication" |
| `3_calculus` | ④ | L22–L26 | "Which way should I nudge the parameters to do a bit better?" — the steering wheel of model learning |
| `4_probability` | ⑤ | L27–L31 | Lets a model say "how sure I am," and puts a computable price on "guessing wrong" |
| `5_audio_dsp` | ⑥ | L32–L53 | Breaking a sound down into a recipe of pure tones (FFT/spectrum/Mel) — the heart of this course |
| `6_deep_learning` | ⑦ | L54–L65 | A stack of functions that tune their own parameters: from hand-written derivatives to training a model that understands yes/no |
| `7_asr` | ⑧ | L66–L75 | Speech to text: cracking the alignment puzzle of "which stretch of audio goes with which word" |
| `8_music` | ⑨ | L76–L82 | Turning a song into a sequence of numbers (an embedding) so a machine can "compare" and "generate" music |
| `9_llm` | ⑩ | L83–L91 | The skeleton of ChatGPT: how a Transformer reads context and how it plugs into outside knowledge (RAG) |
| `10_integration` | ⑪ | L92–L99 | Assembling every earlier part into one complete machine you can demo to the world |

**Two disclaimers**:
1. The later the module, the less you know about it right now, and that's normal — it's fine to be vaguer for `8_music` and `9_llm`;
   even writing "I don't know what an embedding is yet; I'll come back and rewrite this at L76" counts as a passing answer.
   This "purpose list" is a living document: each time you enter a new module, come back and revise its entry.
2. **Where to find the reference answers (setting the rule now so you never have to hunt in two places)**:
   - This lesson's **written examples + complete reference implementations of both functions** are in
     [`solutions/L01_motivation_solutions.en.md`](solutions/L01_motivation_solutions.en.md)
   - Some short in-lesson Q&As live in collapsible `<details>` blocks in the notebook (write your own answer first, then expand)
   - If §8 mentions a "reference implementation" again, it still means `solutions/` by default; the collapsible blocks are in-lesson cheat sheets, not a second answer bank

   When you open the solutions file, **its first section, "Reference implementation 0," shows example answers for this exercise** —
   it has two "concrete" samples, plus one "acceptable vague answer" sample written specifically for unfamiliar modules like `8_music`
   (keep it vague + promise to come back and rewrite it, and that counts as a pass);
   further down, "Reference implementation 1 / 2" hold the reference code for `check_imports` and `environment_report`.


In [ ]:
course_purpose = {
    # ✏️ TODO: replace None with your own understanding (a string)
    # Example: '1_complex_trig': 'I am learning complex numbers and trig so I can understand why FFT output is complex, and what its magnitude and phase each represent'
    '1_complex_trig':   None,
    '2_linear_algebra': None,
    '3_calculus':       None,
    '4_probability':    None,
    '5_audio_dsp':      None,
    '6_deep_learning':  None,
    '7_asr':            None,
    '8_music':          None,
    '9_llm':            None,
    '10_integration':   None,
}

In [ ]:
unfilled = [k for k, v in course_purpose.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
assert all(len(v) > 10 for v in course_purpose.values()), \
    'Each entry should be longer than 10 characters — be more specific'

print('Course purposes recorded:')
for mod, purpose in course_purpose.items():
    print(f'  {mod}: {purpose}')

## 7. Being able to explain it: could you teach each module to someone else?

| The follow-up question | Module |
|---|---|
| Why is the FFT O(N log N)? How do you draw the butterfly network? | L38-L39 |
| Where does the 44100 Hz sample rate (sr) come from? What is aliasing? | L34 |
| Why does the Mel scale match human hearing better than a linear frequency scale? | L46-L47 |
| How does CTC loss handle alignment? What is the blank symbol for? | L68-L69 |
| What's the time complexity of attention? How does a KV-Cache cut down repeated computation? | L83, L85 |
| Why does LoRA reduce the number of parameters? How do you choose the rank? | L84 |
| What metrics do you use to evaluate RAG retrieval quality? | L89 |
| How do you compute INT8 quantization error? How much bigger is the error at 4-bit than at 8-bit? | L87 |

Aurora's goal: **make every row above something you can derive on a whiteboard — not "I've used that library."**

## 8. How to use these notebooks

| Action | Shortcut |
|---|---|
| Run the current cell | `Shift + Enter` |
| Insert a code cell below | `B` |
| Switch to a Markdown cell | `M` |
| Switch to a code cell | `Y` |

**Conventions**:
- A `# ✏️ TODO` marker is a place you need to fill in
- `assert` statements check your implementation automatically; all green means it's correct
- A `# ← change this` marker means a `None` you need to replace
- **Reference answers**: for functions and long-form examples, look in the `solutions/` folder first (for this lesson, `solutions/L01_motivation_solutions.en.md`);
  the in-lesson `<details>` blocks are short "do it first, then peek" answers and don't conflict with solutions — we suggest doing it yourself before looking at either


## 9. ✏️ Implement `check_imports` — debug your environment proactively

Imagine you're a doctor and a patient says, "Something feels off."
You wouldn't say "Well, try this medicine" — you'd run a set of tests first and narrow the problem down to a specific organ.

The same goes for a broken dev environment. "The import failed" is a symptom, not a diagnosis.
`check_imports(names)` is that set of tests: give it a list of package names and it tells you which are installed and which aren't,
pinpointing the problem down to the package instead of leaving you staring blankly at an `ImportError`.

Implement it:
- Use `importlib.util.find_spec(name)` to check each package
- Returns `None` = not installed → `False`; returns anything other than `None` → `True`
- Function signature: `check_imports(names: list[str]) -> dict[str, bool]`

### Dictionaries (dict) in 30 seconds — the only syntax this exercise needs

If your Python stops at lists, spend 30 seconds getting to know dictionaries first: a dictionary is an address book of "name → value."
A list is looked up by position (`lst[0]`); a dictionary is looked up by name (`d['numpy']`). This exercise uses three of its moves:

```python
d = {}                    # Move 1: make an empty address book
d['numpy'] = True         # Move 2: add an entry (key 'numpy', value True)
d['fake_pkg'] = False     #         add another; a repeated key overwrites the old value

for name, ok in d.items():   # Move 3: go through the book entry by entry, getting each key and value together
    print(name, ok)          # prints: numpy True / fake_pkg False
```

Also, the curly-brace form `{'key': None, ...}` is a literal for "adding lots of entries at once" —
it's exactly what the earlier exercise's `course_purpose = {...}` used. The next cell runs a small demo to warm you up.

In [ ]:
# The three dictionary moves · warm-up demo (run it and check the output matches what you expect)
demo = {}                       # Move 1: an empty dictionary
demo['numpy'] = True            # Move 2: add an entry
demo['not_a_real_package'] = False

print('The whole address book:', demo)
print('Look up one entry by name:', demo['numpy'])

for name, ok in demo.items():   # Move 3: loop over keys and values together
    print(f'  {name} -> {ok}')

# Got it? Then you already know the skeleton of check_imports:
# empty dict -> record True/False for each package name -> return the dict

In [ ]:
import importlib.util

def check_imports(names):
    """Check whether a list of packages can be imported. Returns a {package name: bool} dict."""
    # ✏️ TODO: for each name, use importlib.util.find_spec to check whether it exists
    # find_spec returns None = not installed; returns non-None = importable
    result = {}
    raise NotImplementedError("TODO: use importlib.util.find_spec to check each package and return a {name: bool} dict")


In [ ]:
import importlib.util

try:
    status = check_imports(['numpy', 'matplotlib', 'aurora', 'not_a_real_package'])

    assert isinstance(status, dict),                  'check_imports should return a dict'
    assert status.get('numpy') is True,               'numpy should be installed'
    assert status.get('not_a_real_package') is False, 'a package that does not exist should return False'

    print('check_imports test passed')
    print()

    required = ['numpy', 'matplotlib', 'aurora', 'ipykernel']
    result = check_imports(required)
    for name, ok in result.items():
        status_str = 'OK  ' if ok else 'FAIL'
        print(f'  {status_str}  {name}')

except (NotImplementedError, TypeError):
    print('⬜ check_imports is not implemented yet; please fill in the function body')
    print('   Hint: importlib.util.find_spec(name) returning None means not installed')


### When `aurora` comes back False, check these four places

| Symptom | Check first |
|---|---|
| `aurora: False` | Is the kernel name in the top-right corner `Python (AURORA)`? |
| Correct kernel selected but still False | Run `make install` in a terminal, then restart the kernel |
| `make install` fails with an error | Make sure you're in the AURORA root directory and that `pyproject.toml` exists |
| Install succeeded but the import still fails | Run `import sys; print(sys.path)` and confirm `src/` is on the path |

**Why an editable install matters**: `make install` runs
`pip install -e .`, which links `import aurora` straight to `src/aurora/` —
when you change the source you don't have to reinstall; just restart the kernel and the change takes effect.

## 9.5 Smoke test: make the installed aurora package do some real work

> **How to read the number**: 9.5 isn't a "mysterious half-section"; it's an **extension of the §9 environment checkup** —
> once the imports work, you switch on the power and see whether the package starts smoking. When you're done, head on to §10 to write `environment_report`.

After the environment checkup there's one last step: confirm that the installed `aurora` package can actually do work, not just "be imported." In engineering this is called a **smoke test** — when you've assembled a new machine, you first switch on the power and see whether smoke comes out,
before rushing to understand every part.

We'll call `aurora.audio.transforms.fft` on a 4-point signal. **First, let's be clear about what this is**:
the FFT is the star you'll write by hand in L37–L41, and complex numbers are L05's material —
**you're not expected to fully understand this cell's output yet. That's by design; it's not your fault, and it certainly doesn't mean your environment is broken.**

Still, rather than running it with your eyes closed, let's do a round of "guess first, run second, check the answer last."
For that, here's a **minimal concept kit** first (each item is developed properly in later lessons):

- **Sampling point**: the signal `[1, 0, -1, 0]` is just 4 burst-mode snapshots — same idea as the 320 points in Section 1, only short enough to work out in your head.
- **Normalized sample rate sr = 1**: Section 1 used a real sr=16000; for pure math exercises the big numbers get in the way,
  so we simply agree to "take 1 snapshot per second" (normalizing = switching to the most convenient unit). That way point n happens at second n, and the mental arithmetic stays minimal.
- **Fundamental frequency**: in a signal of N points, **the wave that fits exactly 1 complete back-and-forth (one period)**
  is called the signal's fundamental frequency, and numerically it equals sr/N. It's the signal's "frequency ruler": every other frequency is reported as "so many times the fundamental."
- **DC component (DC, frequency = 0 Hz)**: a term borrowed from electronics — direct current doesn't oscillate; it's a flat line,
  so "the component at frequency 0" is the overall level of the signal (its overall offset).
  **That's what the FFT's out[0] holds, and numerically it equals the plain sum of all the sample values** (note: the sum, which is N × the mean, not the mean itself — in a moment you can check this difference yourself).
- **The `j` in the output**: FFT output is made of **complex numbers**, of the form `a+bj` —
  for now, think of j as "the marker for a second coordinate"; `.real` takes the first part, a. L05 explains it fully; today we only look at `.real`.

The next cell first **plots** `[1, 0, -1, 0]` — four isolated numbers don't look like "oscillation," but plot them and you can see it.


In [ ]:
# Plot [1, 0, -1, 0]: 4 points + a cosine curve passing through them
import numpy as np
import matplotlib.pyplot as plt

signal = np.array([1.0, 0.0, -1.0, 0.0])
n = np.arange(4)                       # sample indices 0,1,2,3 (also seconds, since sr=1)
tt = np.linspace(0, 4, 400)            # a dense "continuous" time axis for drawing a smooth curve

plt.figure(figsize=(8, 2.5))
plt.plot(tt, np.cos(2 * np.pi * tt / 4), '--', alpha=0.6,
         label='cos(2*pi*t/4) — exactly 1 complete period in 4 points (= the fundamental)')
plt.stem(n, signal, basefmt=' ', label='4 sample points [1, 0, -1, 0]')
plt.xlabel('Sample index n'); plt.ylabel('Value')
plt.title('1 -> 0 -> -1 -> 0 -> (back to 1): exactly one full back-and-forth')
plt.legend(loc='upper right'); plt.tight_layout(); plt.show()

print('Reading the plot: these 4 points are snapshots of a cosine curve, and the whole signal holds exactly 1 complete period.')
print('In other words, its frequency = 1 × the fundamental. Take this observation with you into the warm-up questions below.')

## ✏️ Warm-up reasoning: think it through before running any code

Before you run the FFT check cell below, answer these two questions (write your answers in the comments of the cell below):

**Q1**: The signal `[1, 0, -1, 0]` has only 4 sample points. What frequency does it represent?
- Sample rate sr = 1 (normalized), signal length N = 4
- Hint: how many complete oscillation periods does this signal go through?

**Q2**: The FFT output `out[0]` is the DC component (DC, frequency = 0 Hz).
What's the mean of `[1, 0, -1, 0]`? What value should out[0].real be close to?

> Write your answers in the comments of the code cell below, then check them once you've run the FFT.

In [ ]:
# ✏️ Write down your predictions (before running; check them afterward)
# Q1 answer: the frequency this signal represents is ______ (how many times the fundamental?)
# Q2 answer: out[0].real should be about ______

# Once your answers are written, run the next cell to check them

In [ ]:
from aurora.audio.transforms import fft
import numpy as np
x = np.array([1.0, 0.0, -1.0, 0.0])
out = fft(x)
assert out.shape == (4,), f"Expected shape (4,), got {out.shape}"
assert np.isclose(out[0].real, 0.0, atol=1e-9), "DC component should be 0"
print("✅ aurora.audio.transforms.fft check passed")

### Checking the answers: what those 4 numbers are telling you

After you run the previous cell, the full contents of `out` are `[0.+0.j, 2.+0.j, 0.+0.j, 2.+0.j]`.
How to read it: **position k of out reports "how strong the component at k times the fundamental is."** One at a time:

- `out[0] = 0`: 0 × the fundamental = no oscillation = the DC component, equal to the sum of all the sample values 1+0+(-1)+0 = 0 ✓
- `out[1] = 2`: **the component at 1 × the fundamental is strong** — which confirms Q1: the signal is one complete period of a wave
- `out[2] = 0`: the component at 2 × the fundamental (2 back-and-forths in 4 points) is zero — the signal contains no such faster wave
- `out[3] = 2`: huh, why is it the same as out[1]?
  This is the **symmetry of real signals** (its formal name, later on, is **conjugate symmetry**):
  for recordings/arrays containing only real numbers, the second half of the spectrum repeats information the first half has already given
  (one way to remember it: `out[N-k]` and `out[k]` come in pairs, like mirror images).
  L38 derives why it has to happen.
  **For today, just remember: out[3] having the same value as out[1] doesn't mean you got it wrong — it's the second half of the ruler mirroring the first half.**

<details>
<summary><b>Reference answers for Q1 / Q2 (write your own first, then expand)</b></summary>

**Q1**: Plot the 4 points (the figure above): 1 → 0 → −1 → 0 → (back to 1),
exactly one complete back-and-forth, so the signal's frequency is **1 × the fundamental** (fundamental = sr/N = 1/4 Hz).
From the output side: `out[1] = 2` is the nonzero peak, which shows that the "1 × fundamental" component is present — that's where you check your answer.

**Q2**: Mean = (1+0−1+0)/4 = **0**. out[0] is the **sum** of all the sample values = 0×4 = **0**,
so `out[0].real ≈ 0`. ⚠️ Be especially careful here: in this example the mean happens to be 0, so "sum" and "mean" give the same number,
and that's pure coincidence — **out[0] always equals the sum (N × mean), not the mean itself**.
Don't believe it? The next cell catches it out on the spot with a signal whose mean isn't 0.

</details>


In [ ]:
# Catching out the "out[0] = mean" coincidence: try two signals whose mean isn't 0
# (A good habit you'll see throughout the course: one example matching doesn't mean the rule is right — build another set of data to check)
for sig in [np.array([1.0, 0.0, -1.0, 0.0]),   # the original problem: mean 0
            np.array([2.0, 1.0, 0.0, 1.0]),    # mean 1: the original signal raised by 1
            np.array([1.0, 1.0, 1.0, 1.0])]:   # pure DC: a flat line
    o = fft(sig)
    print(f'signal {sig}  mean={sig.mean():.1f}  sum={sig.sum():.1f}  '
          f'-> out[0].real={o[0].real:.1f}  full output={np.round(o, 1)}')

print()
print('Conclusion 1: out[0] follows the "sum" (= N × mean), not the mean itself.')
print('Conclusion 2: signal 2 = signal 1 raised by 1 overall, and its output = the original output + the pure-DC output exactly,')
print('        — the FFT keeps separate books for "wiggle" and "overall lift"; that is how it takes sound apart (see L37).')

## 10. ✏️ Implement `environment_report` — record your current runtime environment

During development you often need to tell someone else (or your future self) exactly "what my environment is."
Implement a function that returns a dictionary of environment information — your "environment snapshot."

Keys to include:
- `python_executable`: path to the current Python interpreter (`sys.executable`)
- `python_version`: the version string (`sys.version`)
- `aurora_available`: whether `aurora` can be imported (a boolean)
- `project_root`: path of the directory containing `pyproject.toml` (a string)

### What's that `*` in the hint code? — taking this black box apart in 30 seconds

The hint in the function skeleton below includes the line `for folder in [Path.cwd(), *Path.cwd().parents]:`.
That `*` isn't multiplication (multiplication needs two numbers); it's called **unpacking**:
it "spreads out" a sequence into the list, instead of stuffing the whole sequence in as a single element. Compare:

```python
inner = ['b', 'c']
['a', *inner]   # -> ['a', 'b', 'c']      spread out: 3 elements
['a', inner]    # -> ['a', ['b', 'c']]    stuffed in whole: 2 elements (a list inside a list)
```

`Path.cwd().parents` is the sequence of directories "one level up, two levels up, … all the way to the root," so

```python
[Path.cwd(), *Path.cwd().parents]
# result = [current directory, one level up, two levels up, ..., root]
# a fully equivalent version without the star:
[Path.cwd()] + list(Path.cwd().parents)
```

Pick whichever you like — **if the star confuses you, use the second form**; climbing up one `.parent` at a time in a while loop is perfectly fine too.
This lesson's "no black boxes" spirit applies to every line of hint code as well: if you're not sure what something does, pull it out and run it (the next cell does exactly that).

In [ ]:
# Star unpacking · live check
from pathlib import Path

inner = ['b', 'c']
print('Spread out:', ['a', *inner])     # 3 elements
print('Stuffed in:', ['a', inner])      # 2 elements; the second is the whole list

with_star = [Path.cwd(), *Path.cwd().parents]
no_star   = [Path.cwd()] + list(Path.cwd().parents)
print('\nDo both forms give the same result?', with_star == no_star)
print('The first 3 directories in the sequence (exactly the candidates environment_report checks, level by level):')
for p in with_star[:3]:
    print('  ', p)

In [ ]:
import sys
from pathlib import Path

def environment_report():
    """Return a snapshot dict of the current runtime environment."""
    # ✏️ TODO: fill in the values for the four keys
    #
    # How to implement project_root:
    #   start from Path.cwd() and check each parent directory in turn, moving upward;
    #   the first directory that contains pyproject.toml is the project root.
    #   for folder in [Path.cwd(), *Path.cwd().parents]:
    #       if ...:
    #           root = ...; break
    root = None
    # ← implement the root-finding logic here

    return {
        'python_executable': None,  # ← sys.executable
        'python_version':    None,  # ← sys.version
        'aurora_available':  None,  # ← use check_imports(['aurora'])['aurora']
        'project_root':      None,  # ← use the root found above (as a string)
    }


In [ ]:
import sys
from pathlib import Path

try:
    report = environment_report()

    assert report['python_executable'] == sys.executable,  'python_executable is incorrect'
    assert report['python_version'] == sys.version,        'python_version is incorrect'
    assert isinstance(report['aurora_available'], bool),   'aurora_available should be a boolean'
    assert report['project_root'] is not None,             'project_root was not found'
    assert (Path(report['project_root']) / 'pyproject.toml').exists(), \
        f"project_root={report['project_root']} does not contain pyproject.toml"

    print('environment_report passed')
    print()
    for k, v in report.items():
        val = str(v)[:80] + '...' if len(str(v)) > 80 else str(v)
        print(f'  {k}: {val}')
except (NotImplementedError, TypeError):
    print('⬜ environment_report is not implemented yet; fill in the function body, then run this check again')
except (TypeError, AssertionError) as exc:
    print(f'⚠️ environment_report check did not pass: {exc}')
    print('   Hint: implement the return values for the four keys first, then rerun this cell.')


## 11. ✏️ Write down your 3 learning goals

You just wrote what each module "contributes to Aurora"; now write what it "means to you."
Six months from now, you'll check yourself against these.

In [ ]:
my_goals = {
    # ✏️ TODO: replace None with your goals (strings)
    '技术目标': None,  # technical goal — e.g. 'derive the FFT's O(N log N) on a whiteboard'
    '项目目标': None,  # project goal — e.g. 'put the audio analysis engine on GitHub'
    '面试目标': None,  # interview goal — e.g. 'handle FAANG audio / ML system design interviews'
}

In [ ]:
assert all(v is not None for v in my_goals.values()), 'every goal needs to be filled in'
assert all(isinstance(v, str) and len(v) > 5 for v in my_goals.values()), \
    'goals should be meaningful strings (> 5 characters)'
print('Goals recorded:')
for k, v in my_goals.items():
    print(f'  {k}: {v}')

## Wrap-up

This lesson laid 6 foundations:

1. The Aurora principle: write core capabilities yourself; don't import black boxes (a local pipeline black box is still a black box)
2. A real-understanding mindset: a few deep pieces of work > a big pile of shallow ones (watch out for "fake goal" KPIs)
3. The roadmap: the dependency order of the 11 modules and why you can't skip any; **it's normal that the folders and logical modules don't line up at the very start**
4. Milestones: every month has concrete evidence that "I really got there" (they relate to the four end-state systems of §3 as "milestones vs. destination")
5. `check_imports` / `environment_report`: debug proactively instead of waiting for errors to find you
6. Course purposes + personal goals: internalizing the "why" matters more than the "what"

If you took the opening "30-minute minimum path" to get here: once the sine wave runs, both functions are green, and your three goals are written down, you're ready for L02;
bookmark the roadmap and the milestone table, and pull them out as a checklist at the end of each month.

**Next lesson L02**: sampling and your first audible sine wave — what a sample rate is, and how a numpy array holds a stretch of sound.
You'll write four functions — `samples_count`, `make_time_axis`, `make_sine`, and `signal_summary` —
and every bit of signal processing that follows starts from there.


In [ ]:
# ✏️ Self-review for this lesson — fill it in before running, no peeking at the answers
l01_review = {
    "aurora_no_api_principle": None,   # Can I explain in one sentence why we don't use APIs? True/False
    "roadmap_dependency":      None,   # Do I know why L37-L41 (FFT) can't be skipped? True/False
    "check_imports_done":      None,   # check_imports implemented and passing its tests? True/False
    "environment_report_done": None,   # environment_report implemented and passing its tests? True/False
    "personal_goals_written":  None,   # 3 personal goals written down? True/False
}

unfilled = [k for k, v in l01_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l01_review.items() if v is False]
if weak:
    print(f'⚠️  Go over these items once more: {weak}')
else:
    print('✅ L01 fully cleared! On to L02: the digital representation of sound')

---

→ **Next lesson** · [L02 · Sampling and your first sine wave](L02_sound_digital.en.ipynb)

> Next time: **sampling and your first sine wave** — the sampling theorem, PCM arrays, and synthesizing your first audible sound by hand.